In [ ]:
import pandas as pd
import os
import glob
import zipfile

from google.colab import files

uploaded = files.upload()

Saving olist_customers_dataset.csv to olist_customers_dataset.csv
Saving olist_geolocation_dataset.csv to olist_geolocation_dataset.csv
Saving olist_order_items_dataset.csv to olist_order_items_dataset.csv
Saving olist_order_payments_dataset.csv to olist_order_payments_dataset.csv
Saving olist_order_reviews_dataset.csv to olist_order_reviews_dataset.csv
Saving olist_orders_dataset.csv to olist_orders_dataset.csv
Saving olist_products_dataset.csv to olist_products_dataset.csv
Saving olist_sellers_dataset.csv to olist_sellers_dataset.csv
Saving product_category_name_translation.csv to product_category_name_translation.csv


In [ ]:
import pandas as pd
import io

# Loading all uploaded CSV files into a dictionary
dataframes = {}

for filename, content in uploaded.items():
    if filename.lower().endswith(".csv"):
        dataframes[filename] = pd.read_csv(
            io.BytesIO(content)
        )

print("Total CSV files loaded:", len(dataframes))

# Display row and column counts
for filename, df in dataframes.items():
    print(
        f"{filename}: {df.shape[0]:,} rows, "
        f"{df.shape[1]} columns"
    )

Total CSV files loaded: 9
olist_customers_dataset.csv: 99,441 rows, 5 columns
olist_geolocation_dataset.csv: 1,000,163 rows, 5 columns
olist_order_items_dataset.csv: 112,650 rows, 7 columns
olist_order_payments_dataset.csv: 103,886 rows, 5 columns
olist_order_reviews_dataset.csv: 99,224 rows, 7 columns
olist_orders_dataset.csv: 99,441 rows, 8 columns
olist_products_dataset.csv: 32,951 rows, 9 columns
olist_sellers_dataset.csv: 3,095 rows, 4 columns
product_category_name_translation.csv: 71 rows, 2 columns


In [ ]:
# checking orders table
orders = dataframes["olist_orders_dataset.csv"]

print("First 5 rows:")
display(orders.head())

print("\nColumn information:")
orders.info()

First 5 rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



Column information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [ ]:
# Checking for missing values
for filename, df in dataframes.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    print(f"\n--- {filename} ---")

    if missing.empty:
        print("No missing values found.")
    else:
        print(missing.sort_values(ascending=False))


--- olist_customers_dataset.csv ---
No missing values found.

--- olist_geolocation_dataset.csv ---
No missing values found.

--- olist_order_items_dataset.csv ---
No missing values found.

--- olist_order_payments_dataset.csv ---
No missing values found.

--- olist_order_reviews_dataset.csv ---
review_comment_title      87656
review_comment_message    58247
dtype: int64

--- olist_orders_dataset.csv ---
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64

--- olist_products_dataset.csv ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

--- olist_sellers_dataset.csv ---
No missing values found.

--- product_category_name_translation.csv ---
No missing values found.


In [ ]:
orders = dataframes["olist_orders_dataset.csv"]

# Count orders by status
print("ORDER STATUS BREAKDOWN")
display(
    orders["order_status"]
    .value_counts(dropna=False)
    .rename_axis("Order Status")
    .reset_index(name="Number of Orders")
)

ORDER STATUS BREAKDOWN


,Order Status,Number of Orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [ ]:
# Investigating missing delivery dates by order status

delivery_check = orders.groupby(
    "order_status"
).agg(
    total_orders=("order_id", "count"),
    missing_carrier_date=(
        "order_delivered_carrier_date",
        lambda x: x.isna().sum()
    ),
    missing_delivery_date=(
        "order_delivered_customer_date",
        lambda x: x.isna().sum()
    )
).reset_index()

display(delivery_check)

,order_status,total_orders,missing_carrier_date,missing_delivery_date
0,approved,2,2,2
1,canceled,625,550,619
2,created,5,5,5
3,delivered,96478,2,8
4,invoiced,314,314,314
5,processing,301,301,301
6,shipped,1107,0,1107
7,unavailable,609,609,609


In [ ]:
# Checking for duplicate records
tables_to_check = [
    "olist_orders_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_order_reviews_dataset.csv"
]

for name in tables_to_check:
    df = dataframes[name]

    print(f"\n{name}")
    print("Duplicate full rows:", df.duplicated().sum())

    # Check the expected identifier for each table
    id_columns = {
        "olist_orders_dataset.csv": "order_id",
        "olist_customers_dataset.csv": "customer_id",
        "olist_products_dataset.csv": "product_id",
        "olist_sellers_dataset.csv": "seller_id",
        "olist_order_reviews_dataset.csv": "review_id"
    }

    key = id_columns[name]
    print(f"Duplicate {key} values:", df[key].duplicated().sum())


olist_orders_dataset.csv
Duplicate full rows: 0
Duplicate order_id values: 0

olist_customers_dataset.csv
Duplicate full rows: 0
Duplicate customer_id values: 0

olist_products_dataset.csv
Duplicate full rows: 0
Duplicate product_id values: 0

olist_sellers_dataset.csv
Duplicate full rows: 0
Duplicate seller_id values: 0

olist_order_reviews_dataset.csv
Duplicate full rows: 0
Duplicate review_id values: 814


In [ ]:
# Checking whether dates are valid
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_check = orders.copy()

for col in date_columns:
    orders_check[col] = pd.to_datetime(
        orders_check[col],
        errors="coerce"
    )

print("Date conversion check:")

for col in date_columns:
    print(
        col,
        "| Missing or invalid dates:",
        orders_check[col].isna().sum()
    )

Date conversion check:
order_purchase_timestamp | Missing or invalid dates: 0
order_approved_at | Missing or invalid dates: 160
order_delivered_carrier_date | Missing or invalid dates: 1783
order_delivered_customer_date | Missing or invalid dates: 2965
order_estimated_delivery_date | Missing or invalid dates: 0


In [ ]:
# Investigating the repeated review IDs
reviews = dataframes["olist_order_reviews_dataset.csv"]

# Findng all rows whose review ID appears more than once
duplicate_reviews = reviews[
    reviews.duplicated("review_id", keep=False)
].sort_values("review_id")

print("Rows with repeated review IDs:", len(duplicate_reviews))
print(
    "Unique repeated review IDs:",
    duplicate_reviews["review_id"].nunique()
)

# Checking whether the same review ID is linked to multiple orders
review_id_check = reviews.groupby("review_id").agg(
    number_of_rows=("review_id", "size"),
    unique_orders=("order_id", "nunique")
).reset_index()

display(
    review_id_check[
        review_id_check["number_of_rows"] > 1
    ].head(10)
)

# Inspect the actual records
display(duplicate_reviews.head(10))

Rows with repeated review IDs: 1603
Unique repeated review IDs: 789


,review_id,number_of_rows,unique_orders
31,00130cbe1f9d422698c812ed8ded1919,2,2
425,0115633a9c298b6a98bcbe4eee75345f,2,2
568,0174caf0ee5964646040cd94e15ac95e,2,2
574,017808d29fd1f942d97e50184dfb4c13,2,2
884,0254bd905dc677a6078990aad3331a36,2,2
961,0288d42bef3dfe36930740c9588a570f,2,2
1000,02aa7f5f75e964e3c7efa59a1f515281,2,2
1225,034528cb00b6bc981847acafbdf2ae0b,2,2
1364,03a6a25db577d0689440933055111897,2,2
1674,0467560f511c516ddaa54a60edb0c291,2,2


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [ ]:
# Investigating the eight delivered orders with missing delivery dates
orders = dataframes["olist_orders_dataset.csv"]

# Orders marked delivered but missing actual delivery dates
delivered_missing_dates = orders[
    (orders["order_status"] == "delivered") &
    (
        orders["order_delivered_customer_date"].isna() |
        orders["order_delivered_carrier_date"].isna()
    )
]

display(delivered_missing_dates)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaN,2017-11-20 19:44:47,2017-11-14 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [ ]:
# Creating  first cleaned analysis table
import pandas as pd

orders_clean = orders.copy()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col],
        errors="coerce"
    )

# Calculate actual delivery duration in days
orders_clean["delivery_days"] = (
    orders_clean["order_delivered_customer_date"] -
    orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Calculate days early or late relative to the estimated date
orders_clean["delivery_variance_days"] = (
    orders_clean["order_delivered_customer_date"] -
    orders_clean["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

# Positive = late; zero or negative = on time or early
orders_clean["delivery_status"] = "Not assessable"

valid_delivery = (
    orders_clean["order_status"].eq("delivered") &
    orders_clean["order_delivered_customer_date"].notna() &
    orders_clean["order_estimated_delivery_date"].notna()
)

orders_clean.loc[valid_delivery, "delivery_status"] = (
    orders_clean.loc[valid_delivery, "delivery_variance_days"]
    .gt(0)
    .map({True: "Late", False: "On time"})
)

display(
    orders_clean[
        [
            "order_id",
            "order_status",
            "delivery_days",
            "delivery_variance_days",
            "delivery_status"
        ]
    ].head(10)
)

,order_id,order_status,delivery_days,delivery_variance_days,delivery_status
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,8.436574,-7.107488,On time
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,13.782037,-5.355729,On time
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,9.394213,-17.245498,On time
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,13.208750,-12.980069,On time
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2.873877,-9.238171,On time
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,16.542245,-5.543113,On time
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,NaN,NaN,Not assessable
7,6514b8ad8028c9f2cc2374ded245783f,delivered,9.989826,-11.461215,On time
8,76c6e866289321a7c93b82b54852dc33,delivered,9.818762,-31.410995,On time
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,18.221852,-6.281597,On time


In [ ]:
# Calculating the average delivery time, on-time delivery rate, and late-delivery rate.
# Making a separate copy so the original dataset stays unchanged
orders_kpi = orders_clean.copy()

# Comparing calendar dates, rather than exact times
orders_kpi["delivery_variance_calendar_days"] = (
    orders_kpi["order_delivered_customer_date"].dt.normalize()
    - orders_kpi["order_estimated_delivery_date"].dt.normalize()
).dt.days

# Only delivered orders with both required dates can be assessed
assessable = (
    orders_kpi["order_status"].eq("delivered")
    & orders_kpi["order_delivered_customer_date"].notna()
    & orders_kpi["order_estimated_delivery_date"].notna()
)

# Classify delivery performance
orders_kpi["delivery_status_calendar"] = "Not assessable"

orders_kpi.loc[
    assessable & (orders_kpi["delivery_variance_calendar_days"] > 0),
    "delivery_status_calendar"
] = "Late"

orders_kpi.loc[
    assessable & (orders_kpi["delivery_variance_calendar_days"] <= 0),
    "delivery_status_calendar"
] = "On time"

# Calculating  KPIs
delivered_orders = orders_kpi[
    orders_kpi["order_status"] == "delivered"
]

assessable_orders = orders_kpi[assessable]

on_time_count = (
    assessable_orders["delivery_status_calendar"] == "On time"
).sum()

late_count = (
    assessable_orders["delivery_status_calendar"] == "Late"
).sum()

average_delivery_days = delivered_orders["delivery_days"].mean()

on_time_rate = on_time_count / len(assessable_orders) * 100
late_rate = late_count / len(assessable_orders) * 100

print("Total orders:", len(orders_kpi))
print("Delivered orders:", len(delivered_orders))
print("Delivered orders assessable for delivery performance:",
      len(assessable_orders))
print("Delivered orders not assessable:",
      len(delivered_orders) - len(assessable_orders))
print("Average delivery time (days):", round(average_delivery_days, 2))
print("On-time delivery count:", on_time_count)
print("Late delivery count:", late_count)
print("On-time delivery rate (%):", round(on_time_rate, 2))
print("Late delivery rate (%):", round(late_rate, 2))

Total orders: 99441
Delivered orders: 96478
Delivered orders assessable for delivery performance: 96470
Delivered orders not assessable: 8
Average delivery time (days): 12.56
On-time delivery count: 89936
Late delivery count: 6534
On-time delivery rate (%): 93.23
Late delivery rate (%): 6.77


In [ ]:

reviews = dataframes["olist_order_reviews_dataset.csv"]

print(
    "Repeated review_id + order_id rows:",
    reviews.duplicated(
        subset=["review_id", "order_id"],
        keep=False
    ).sum()
)

print(
    "Repeated order_id rows:",
    reviews["order_id"].duplicated(keep=False).sum()
)

Repeated review_id + order_id rows: 0
Repeated order_id rows: 1098


In [ ]:

reviews = dataframes["olist_order_reviews_dataset.csv"]

# Count reviews for each order
reviews_per_order = reviews.groupby("order_id").agg(
    review_count=("review_id", "size"),
    different_scores=("review_score", "nunique")
).reset_index()

# Keep orders that have more than one review record
multiple_reviews = reviews_per_order[
    reviews_per_order["review_count"] > 1
]

print("Orders with multiple review records:",
      len(multiple_reviews))

print("Orders with different scores:",
      (multiple_reviews["different_scores"] > 1).sum())

display(multiple_reviews.head(10))

Orders with multiple review records: 547
Orders with different scores: 202


,order_id,review_count,different_scores
84,0035246a40f520710769010f752e7507,2,1
461,013056cfe49763c6f66bda03396c5ee3,2,2
556,0176a6846bcb3b0d3aa3116a9a768597,2,1
835,02355020fd0a40a0d56df9f6ff060413,2,2
985,029863af4b968de1e5d6a82782e662f5,2,2
1092,02e0b68852217f5715fb9cc885829454,2,1
1103,02e723e8edb4a123d414f56cc9c4665e,2,1
1272,03515a836bb855b03f7df9dee520a8fc,2,1
1455,03c939fd7fd3b38f8485a0f95798f1f6,3,2
1510,03eba6d9fef8f5b3e811d4b5a7cca9cd,2,2


**DATA** **CLEANING**




In [ ]:

import pandas as pd

# Create copies of the original datasets
customers_clean = dataframes["olist_customers_dataset.csv"].copy()
orders_clean = dataframes["olist_orders_dataset.csv"].copy()
items_clean = dataframes["olist_order_items_dataset.csv"].copy()
payments_clean = dataframes["olist_order_payments_dataset.csv"].copy()
reviews_clean = dataframes["olist_order_reviews_dataset.csv"].copy()
products_clean = dataframes["olist_products_dataset.csv"].copy()
sellers_clean = dataframes["olist_sellers_dataset.csv"].copy()

# Convert order date columns into proper datetime format
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_date_columns:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col], errors="coerce"
    )

print("Clean copies created successfully!")

print("Orders:", orders_clean.shape)
print("Customers:", customers_clean.shape)
print("Order items:", items_clean.shape)
print("Payments:", payments_clean.shape)
print("Reviews:", reviews_clean.shape)
print("Products:", products_clean.shape)
print("Sellers:", sellers_clean.shape)

Clean copies created successfully!
Orders: (99441, 8)
Customers: (99441, 5)
Order items: (112650, 7)
Payments: (103886, 5)
Reviews: (99224, 7)
Products: (32951, 9)
Sellers: (3095, 4)


In [ ]:

# Load the category translation table
category_translation = dataframes[
    "product_category_name_translation.csv"
].copy()

# See the available translations
display(category_translation.head(10))

print("Number of category translations:",
      len(category_translation))

# Add English category names to the products table
products_clean = products_clean.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

# Use the English name where available.
# Otherwise, retain the original category name.
products_clean["product_category_english"] = (
    products_clean["product_category_name_english"]
    .fillna(products_clean["product_category_name"])
)

print("Products table after translation:", products_clean.shape)

display(
    products_clean[
        ["product_id", "product_category_name",
         "product_category_english"]
    ].head(10)
)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


Number of category translations: 71
Products table after translation: (32951, 11)


,product_id,product_category_name,product_category_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares
5,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,musical_instruments
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,cool_stuff
7,2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,furniture_decor
8,37cc742be07708b53a98702e77a21a02,eletrodomesticos,home_appliances
9,8c92109888e8cdf9d66dc7e463025574,brinquedos,toys


In [ ]:

# Check whether each order's customer exists in customers
print(
    "Orders with unknown customer IDs:",
    (~orders_clean["customer_id"].isin(
        customers_clean["customer_id"]
    )).sum()
)

# Check whether order items reference existing orders
print(
    "Order items with unknown order IDs:",
    (~items_clean["order_id"].isin(
        orders_clean["order_id"]
    )).sum()
)

# Check whether order items reference existing products
print(
    "Order items with unknown product IDs:",
    (~items_clean["product_id"].isin(
        products_clean["product_id"]
    )).sum()
)

# Check whether order items reference existing sellers
print(
    "Order items with unknown seller IDs:",
    (~items_clean["seller_id"].isin(
        sellers_clean["seller_id"]
    )).sum()
)

# Check payments and reviews against existing orders
print(
    "Payments with unknown order IDs:",
    (~payments_clean["order_id"].isin(
        orders_clean["order_id"]
    )).sum()
)

print(
    "Reviews with unknown order IDs:",
    (~reviews_clean["order_id"].isin(
        orders_clean["order_id"]
    )).sum()
)

Orders with unknown customer IDs: 0
Order items with unknown order IDs: 0
Order items with unknown product IDs: 0
Order items with unknown seller IDs: 0
Payments with unknown order IDs: 0
Reviews with unknown order IDs: 0


In [ ]:

# 1. Calculating item-level totals for each order
items_summary = (
    items_clean.groupby("order_id")
    .agg(
        total_items=("order_item_id", "count"),
        item_sales=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

# 2. Calculating payment totals for each order
payments_summary = (
    payments_clean.groupby("order_id")
    .agg(
        total_payment=("payment_value", "sum"),
        payment_records=("payment_sequential", "count")
    )
    .reset_index()
)

# 3. Joining summaries to orders
orders_analysis = orders_clean.merge(
    items_summary,
    on="order_id",
    how="left",
    validate="one_to_one"
)

orders_analysis = orders_analysis.merge(
    payments_summary,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Orders analysis table:", orders_analysis.shape)
print("Unique order IDs:", orders_analysis["order_id"].nunique())

display(
    orders_analysis[
        ["order_id", "order_status", "total_items",
         "item_sales", "freight_value", "total_payment"]
    ].head(10)
)


Orders analysis table: (99441, 13)
Unique order IDs: 99441


,order_id,order_status,total_items,item_sales,freight_value,total_payment
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,1.0,29.99,8.72,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,1.0,118.70,22.76,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,1.0,159.90,19.22,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,1.0,45.00,27.20,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,1.0,19.90,8.72,28.62
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,1.0,147.90,27.36,175.26
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,1.0,49.90,16.05,65.95
7,6514b8ad8028c9f2cc2374ded245783f,delivered,1.0,59.99,15.17,75.16
8,76c6e866289321a7c93b82b54852dc33,delivered,1.0,19.90,16.05,35.95
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,1.0,149.99,19.77,169.76


In [ ]:

print("Orders in original table:", len(orders_clean))
print("Rows in analysis table:", len(orders_analysis))
print("Duplicate order IDs:", orders_analysis["order_id"].duplicated().sum())

print(
    "Orders with no item summary:",
    orders_analysis["total_items"].isna().sum()
)

print(
    "Orders with no payment summary:",
    orders_analysis["total_payment"].isna().sum()
)


Orders in original table: 99441
Rows in analysis table: 99441
Duplicate order IDs: 0
Orders with no item summary: 775
Orders with no payment summary: 1


In [ ]:

# Check orders without item records
missing_items = orders_analysis[
    orders_analysis["total_items"].isna()
]

print("Orders without item records by status:")
display(missing_items["order_status"].value_counts().to_frame(
    name="order_count"
))

# Check orders without payment records
missing_payments = orders_analysis[
    orders_analysis["total_payment"].isna()
]

print("Orders without payment records by status:")
display(missing_payments["order_status"].value_counts().to_frame(
    name="order_count"
))

# Inspect a few orders without item records
display(
    missing_items[
        ["order_id", "order_status", "order_purchase_timestamp"]
    ].head(10)
)


Orders without item records by status:


,order_count
order_status,
unavailable,603
canceled,164
created,5
invoiced,2
shipped,1


Orders without payment records by status:


,order_count
order_status,
delivered,1


,order_id,order_status,order_purchase_timestamp
266,8e24261a7e58791d10cb1bf9da94df5c,unavailable,2017-11-16 15:09:28
586,c272bcd21c287498b4883c7512019702,unavailable,2018-01-31 11:31:37
687,37553832a3a89c9b2db59701c357ca67,unavailable,2017-08-14 17:38:02
737,d57e15fb07fd180f06ab3926b39edcd2,unavailable,2018-01-08 19:39:03
1130,00b1cb0320190ca0daa2c88b35206009,canceled,2018-08-28 15:26:39
1160,2f634e2cebf8c0283e7ef0989f77d217,unavailable,2017-09-27 20:55:33
1579,ee0db22a8e742b752914016708470ec8,unavailable,2017-08-24 11:04:41
1801,ed3efbd3a87bea76c2812c66a0b32219,canceled,2018-09-20 13:54:16
1826,6ad57aecbae806a7e9cc2cdb6b380711,unavailable,2017-11-30 07:48:24
1868,df8282afe61008dc26c6c31011474d02,canceled,2017-03-04 12:14:30


In [ ]:

# Inspect orders without item records that have unusual statuses
inspect_statuses = ["invoiced", "shipped"]

display(
    missing_items[
        missing_items["order_status"].isin(inspect_statuses)
    ][
        ["order_id", "order_status", "order_purchase_timestamp",
         "order_approved_at", "order_delivered_carrier_date",
         "order_delivered_customer_date"]
    ]
)

# Inspect the delivered order without a payment record
display(
    missing_payments[
        ["order_id", "order_status", "order_purchase_timestamp",
         "order_delivered_customer_date", "total_items", "item_sales"]
    ]
)


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
23254,a68ce1686d536ca72bd2dadc4b8671e5,shipped,2016-10-05 01:47:40,2016-10-07 03:11:22,2016-11-07 16:37:37,NaT
57591,2ce9683175cdab7d1c95bcbb3e36f478,invoiced,2016-10-05 21:03:33,2016-10-06 07:46:39,NaT,NaT
69926,e04f1da1f48bf2bbffcf57b9824f76e1,invoiced,2016-10-05 13:22:20,2016-10-06 15:51:38,NaT,NaT


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,total_items,item_sales
30710,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,2016-11-09 07:47:38,3.0,134.97


In [ ]:

# Select delivered orders that have item records
sales_data = orders_analysis[
    (orders_analysis["order_status"] == "delivered")
    & (orders_analysis["item_sales"].notna())
].copy()

# Create a month field from the purchase date
sales_data["order_month"] = (
    sales_data["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

# Summarize by month
monthly_sales = (
    sales_data.groupby("order_month")
    .agg(
        delivered_orders=("order_id", "nunique"),
        product_sales=("item_sales", "sum")
    )
    .reset_index()
    .sort_values("order_month")
)

monthly_sales["product_sales"] = monthly_sales["product_sales"].round(2)

display(monthly_sales)


,order_month,delivered_orders,product_sales
0,2016-09,1,134.97
1,2016-10,265,40325.11
2,2016-12,1,10.90
3,2017-01,750,111798.36
4,2017-02,1653,234223.40
5,2017-03,2546,359198.85
6,2017-04,2303,340669.68
7,2017-05,3546,489338.25
8,2017-06,3135,421923.37
9,2017-07,3872,481604.52


In [ ]:

# Step 1: Keep delivered orders
delivered_orders = orders_clean[
    orders_clean["order_status"] == "delivered"
][["order_id"]].copy()

# Step 2: Connect order items with delivered orders
category_data = items_clean.merge(
    delivered_orders,
    on="order_id",
    how="inner"
)

# Step 3: Add product category information
category_data = category_data.merge(
    products_clean[
        ["product_id", "product_category_english"]
    ],
    on="product_id",
    how="left"
)

# Step 4: Calculate sales by product category
category_sales = (
    category_data.groupby("product_category_english")
    .agg(
        total_orders=("order_id", "nunique"),
        items_sold=("order_item_id", "count"),
        product_sales=("price", "sum")
    )
    .reset_index()
    .sort_values("product_sales", ascending=False)
)

category_sales["product_sales"] = category_sales["product_sales"].round(2)

# Step 5: Display the top 10 categories
display(category_sales.head(10))


,product_category_english,total_orders,items_sold,product_sales
43,health_beauty,8647,9465,1233131.72
72,watches_gifts,5495,5859,1166176.98
7,bed_bath_table,9272,10953,1023434.76
67,sports_leisure,7530,8431,954852.55
15,computers_accessories,6530,7644,888724.61
39,furniture_decor,6307,8160,711927.69
49,housewares,5743,6795,615628.69
20,cool_stuff,3559,3718,610204.10
5,auto,3810,4140,578966.65
71,toys,3804,4030,471286.48
